In [ ]:
from nautilus_trader.backtest.engine import BacktestEngine
help(BacktestEngine)


In [ ]:
import os
import pandas as pd
from datetime import datetime

# Nautilus imports for constructing data and instruments
from nautilus_trader.model.data import QuoteTick
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Price, Quantity, Currency
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog
from nautilus_trader.model.instruments.option_contract import OptionContract
from nautilus_trader.model.identifiers import InstrumentId, Symbol, Venue

# --- Set directories
csv_dir = "data/nse"
catalog_dir = "catalog"
catalog_meta_dir = "catalog-meta"
os.makedirs(catalog_dir, exist_ok=True)
os.makedirs(catalog_meta_dir, exist_ok=True)

# --- Initialize Parquet catalog to store core data (quote ticks and instruments)
catalog = ParquetDataCatalog(catalog_dir)

# --- Storage lists
all_ticks = []                 # Stores QuoteTick instances
instrument_ids = set()         # Keeps track of all unique instrument IDs
meta_records = []              # Stores auxiliary/meta data as dict rows

# --- Loop through each CSV file in the input directory
for fname in os.listdir(csv_dir):
    if not fname.endswith(".csv") or fname.startswith("._"):
        continue

    print(f"Reading {fname}")
    df = pd.read_csv(os.path.join(csv_dir, fname))

    # Convert timestamps to nanoseconds since epoch (UNIX time)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%y %H:%M")
    df["timestamp"] = df["timestamp"].apply(dt_to_unix_nanos)

    # --- Parse rows into QuoteTick objects
    for _, row in df.iterrows():
        try:
             
            iid_str = row["symbol"].strip()
            parts = iid_str.split(".")
            symbol = parts[0]               # e.g. BANKNIFTY
            venue = parts[1]                # e.g. NSE
            type = parts[2]                 # e.g. OPTION
            expiry_raw = parts[3]           # e.g. 26Jun2025
            strike = float(parts[4])
            right = parts[5].upper()        # CALL or PUT
            symbolplus = f"{symbol}.{type}.{expiry_raw}.{int(strike)}.{right}"
            instrument_id = InstrumentId(symbol=Symbol(symbolplus), venue=Venue(venue))

            # Setup price and size values
            bid_val = float(row["bid"])
            ask_val = float(row["ask"])
            bid_precision = 2
            ask_precision = 2
            bid_size_val = float(row.get("bid_size", 1))
            ask_size_val = float(row.get("ask_size", 1))
            size_precision = 0

            # Create QuoteTick object
            tick = QuoteTick(
                instrument_id=instrument_id,
                ts_event=int(row["timestamp"]),
                ts_init=int(row["timestamp"]),
                bid_price=Price(int(bid_val * 10**bid_precision), bid_precision),
                ask_price=Price(int(ask_val * 10**ask_precision), ask_precision),
                bid_size=Quantity(int(bid_size_val * 10**size_precision), size_precision),
                ask_size=Quantity(int(ask_size_val * 10**size_precision), size_precision),
            )

            all_ticks.append(tick)
          
            instrument_ids.add(instrument_id)

            # Save the meta fields separately
            meta_records.append({
                "instrument_id": f"{symbolplus}.{venue}",
                "timestamp": row["timestamp"],
                "impliedVolatility": row.get("impliedVolatility"),
                "openInterest": row.get("openInterest"),
                "last": row.get("last"),
                "pChange": row.get("pChange"),
            })

        except Exception as e:
            print(f"⚠️ Skipping row due to tick parsing error: {e}")

# --- Build dummy instruments using OptionContract.from_dict
# This will store instrument metadata needed by the engine
print(list(instrument_ids)[0])
print(list(meta_records)[0])
print(all_ticks[0])


In [ ]:
from nautilus_trader.model.instruments.option_contract import OptionContract
from nautilus_trader.model.identifiers import InstrumentId, Symbol, Venue
from nautilus_trader.model.enums import AssetClass, OptionKind
from nautilus_trader.model.objects import Price, Quantity, Currency
from nautilus_trader.core.datetime import dt_to_unix_nanos
from datetime import datetime, timezone

# Example loop for just 1 instrument ID
dummy_instruments = []
for iid in instrument_ids:
    try:
        print(f"🔍 Processing: {iid.value}")
        parts = iid.value.split(".")
        symbol = parts[0]               # e.g. BANKNIFTY
        type = parts[1] 
        expiry_raw = parts[2]           # e.g. 26Jun2025
        strike = float(parts[3])
        right = parts[4].upper()        # CALL or PUT
        venue = parts[5]               # e.g. NSE
        symbolplus = f"{symbol}.{type}.{expiry_raw}.{int(strike)}.{right}"

        expiry_dt = datetime.strptime(expiry_raw, "%d%b%Y")
        expiry_utc = expiry_dt.replace(hour=15, minute=30, tzinfo=timezone.utc)
        now_utc = datetime.now(timezone.utc)

        option = OptionContract(
            instrument_id=InstrumentId(symbol=Symbol(symbolplus ), venue=Venue(venue)),
            raw_symbol=Symbol(symbol),
            asset_class=AssetClass.INDEX,     # Same as example
            exchange=venue,
            currency=Currency.from_str("INR"),
            price_precision=2,
            price_increment=Price.from_str("0.05"),
            multiplier=Quantity.from_int(15),
            lot_size=Quantity.from_int(1),
            underlying=f"{symbol}.{venue}.INDEX",
            option_kind=OptionKind[right],
            strike_price=Price.from_str(str(strike)),
            activation_ns=dt_to_unix_nanos(now_utc),
            expiration_ns=dt_to_unix_nanos(expiry_utc),
            ts_event=dt_to_unix_nanos(now_utc),
            ts_init=dt_to_unix_nanos(now_utc),
        )

        print(f"✅ Created OptionContract: {option.id}")
        dummy_instruments.append(option)
    except Exception as e:
        print(f"⚠️ Skipping {iid.value} due to instrument creation error: {e}")

catalog.write_data(dummy_instruments)


In [ ]:
# --- Write core data to catalog directory
catalog.write_data(dummy_instruments)
catalog.write_data(all_ticks)

# --- Write meta-data (IV, OI, etc.) to separate catalog-meta folder
meta_df = pd.DataFrame(meta_records)
meta_df.to_parquet(os.path.join(catalog_meta_dir, "tick_metadata.parquet"), index=False)

print(f"\n✅ Written {len(all_ticks)} quote ticks to {catalog_dir}/")
print(f"✅ Written {len(meta_df)} metadata records to {catalog_meta_dir}/")
print(f"✅ Written {len(dummy_instruments)} dummy instruments")

In [ ]:
import logging

# Enable debug-level logs
logging.basicConfig(level=logging.DEBUG)


In [ ]:
from decimal import Decimal
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig
from nautilus_trader.model import InstrumentId, TraderId
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.objects import Money
from nautilus_trader.model.currencies import INR
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog

from src.strategies.my_nse_strategy import MyNSEStrategy, MyNSEStrategyConfig

# Step 1: Backtest engine config with debug logging
engine_config = BacktestEngineConfig(
    trader_id=TraderId("BACKTEST_TRADER-001"),
    logging=LoggingConfig(log_level="DEBUG"),
)

engine = BacktestEngine(config=engine_config)






In [ ]:
# Step 2: Load instruments and quote ticks from catalog
data_catalog = ParquetDataCatalog("catalog")  # folder with .parquet files

instruments = data_catalog.instruments()
print(data_catalog.list_data_types())


In [ ]:

print(f"Loaded {len(instruments)} instruments")
ticks = data_catalog.quote_ticks()


In [ ]:
print("🔎 Unique instrument IDs in ticks:")
unique_ids = set(str(tick.instrument_id) for tick in ticks)
for iid in sorted(unique_ids):
    print(iid)


In [ ]:
print("🛠️ Instruments being added to the engine:")
for inst in instruments[:4]:
    print(inst)



In [ ]:
# Step 3: Add venue, instrument, and data to engine
venue_id = "NSE"
existing_venues = [str(v) for v in engine.list_venues()]
print ("Why")
if venue_id not in existing_venues:
    engine.add_venue(
        venue=Venue(venue_id),
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, INR)],
        base_currency=INR,
        default_leverage=Decimal(1),
    )


In [ ]:
# Add all instruments
for inst in instruments:
    try:
        engine.add_instrument(inst)
    except Exception as e:
        print(f"⚠️ Error adding instrument {inst.instrument_id}: {e}")
print("✅ Added  engine.")
# Add all quote ticks (can be filtered if needed)
try:
    engine.add_data(ticks)
    print(f"✅ Added {len(ticks)} quote ticks to the engine.")
except Exception as e:
    print(f"⚠️ Error adding quote ticks: {e}")


In [ ]:

# Step 4: Add your strategy
import sys
sys.path.append('src')
import yaml
from strategies.my_nse_strategy import MyNSEStrategy, MyNSEStrategyConfig
from nautilus_trader.model import InstrumentId

# Load YAML
with open("src/config/nse_strategy_config.yaml") as f:
    yaml_config = yaml.safe_load(f)

# Create config, using YAML values if present, else class defaults
config = MyNSEStrategyConfig(
    instrument_id=InstrumentId.from_str(yaml_config.get("instrument_id", "BANKNIFTY.OPT.26Jun2025.53300.CALL.NSE")),
    sl_pct=yaml_config.get("sl_pct", MyNSEStrategyConfig.sl_pct),
    tp_pct=yaml_config.get("tp_pct", MyNSEStrategyConfig.tp_pct),
    atm_window=yaml_config.get("atm_window", MyNSEStrategyConfig.atm_window),
    position_size=yaml_config.get("position_size", MyNSEStrategyConfig.position_size),
    end_time=yaml_config.get("end_time", MyNSEStrategyConfig.end_time),
    entry_buffer_pct=yaml_config.get("entry_buffer_pct", MyNSEStrategyConfig.entry_buffer_pct),
    lookback_intervals=yaml_config.get("lookback_intervals", MyNSEStrategyConfig.lookback_intervals),
    min_iv=yaml_config.get("min_iv", MyNSEStrategyConfig.min_iv),
    min_oi_change=yaml_config.get("min_oi_change", MyNSEStrategyConfig.min_oi_change),
    breakeven_trigger_pct=yaml_config.get("breakeven_trigger_pct", MyNSEStrategyConfig.breakeven_trigger_pct),
    sar_enabled=yaml_config.get("sar_enabled", MyNSEStrategyConfig.sar_enabled),
)
strategy = MyNSEStrategy(config)
engine.add_strategy(strategy)
  


In [ ]:
engine.run()






In [ ]:
engine.reset()

In [ ]:
engine.reset()

In [ ]:
engine.dispose()

In [ ]:
# View trades
strategy.trades[:5]

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# Your data
data = {
    "instrument_id": ["BANKNIFTY.NSE.OPT.26Jun2025.53300.CALL"] * 3,
    "bid": [100.0, 101.0, 102.0],
    "ask": [102.0, 103.0, 104.0],
    "bid_size": [1, 1, 1],
    "ask_size": [1, 1, 1],
    "timestamp": [1750248000000000000, 1750248900000000000, 1750249800000000000],
}
df = pd.DataFrame(data)
df["ts_event"] = df["timestamp"]
df["ts_init"] = df["timestamp"]
df["venue_id"] = "NSE"

# **Add the type column**
df["type"] = "QUOTE_TICK"

# Save to Parquet with the correct name
parquet_dir = "catalog"
os.makedirs(parquet_dir, exist_ok=True)
parquet_path = os.path.join(parquet_dir, "BANKNIFTY.NSE.OPT.26Jun2025.53300.CALL.pq")
table = pa.Table.from_pandas(df)
pq.write_table(table, parquet_path)

In [ ]:
  df = pd.read_parquet(parquet_path)
  print(df.head())
  print(df.dtypes)
  print(df["type"].unique())

In [ ]:
from nautilus_trader.config import DataCatalogConfig

catalog_config = DataCatalogConfig(
    path="catalog",  # directory, not file
    # other config options as needed
)

from nautilus_trader.config import BacktestEngineConfig

engine_config = BacktestEngineConfig(
    catalogs=[catalog_config],
    # ... other config options as needed
)

from nautilus_trader.backtest.engine  import BacktestEngine

engine = BacktestEngine(engine_config)

In [ ]:
from nautilus_trader.model.identifiers import StrategyId

strategy = MyNSEStrategy(
    config,
    strategy_id=StrategyId("MyNSEStrategy-001")  # Use a unique ID
)
engine.add_strategy(strategy)

In [ ]:
engine.run()

In [ ]:
# 5. Analyze results
# You can now access engine.orders, engine.trades, engine.portfolio, etc.
print("Orders:", engine.orders)
print("Trades:", engine.trades)
print("Portfolio:", engine.portfolio)

In [ ]:
import nautilus_trader.model.identifiers as ids
print(dir(ids))